In [ ]:
from censusdis.states import STATE_CO
from basket.access_acs import dwnld_acs5, make_race_tables, \
    make_pov_limit_tables, make_age_sex_tables, \
        make_units_in_struct_tables, make_year_built_tables, \
            make_civ_emp_tables, make_tenure_tables, \
                make_med_inc_tables, make_occupancy_tables, \
                    make_basics_tables
from basket.export_acs import acs2parquet
import pandas as pd
from pathlib import Path

%load_ext autoreload
%autoreload 2

In [ ]:
def census_prep(state: str, in_path: Path):
    for geos in ["tract", "place", "county"]:
        rt = make_race_tables(state=state, geo=geos)
        pov = make_pov_limit_tables(state=state, geo=geos)
        age_sex = make_age_sex_tables(state=state, geo=geos)
        units = make_units_in_struct_tables(state=state, geo=geos)
        yrblt = make_year_built_tables(state=state, geo=geos)
        emp = make_civ_emp_tables(state=state, geo=geos)
        ten = make_tenure_tables(state=state, geo=geos)
        med_inc = make_med_inc_tables(state=state, geo=geos)
        occ = make_occupancy_tables(state=state, geo=geos)
        basics = make_basics_tables(state=state, geo=geos)

        merge_cols = ["NAME", "STATE", "COUNTY", "TRACT", "PLACE", "GEOID", "geometry"]

        rt_pov = pd.merge(
            left=rt,
            right=pov,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )
        asu = pd.merge(
            left=age_sex,
            right=units,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )
        ybe = pd.merge(
            left=yrblt,
            right=emp,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )
        tmi = pd.merge(
            left=ten,
            right=med_inc,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )
        otmi = pd.merge(
            left=tmi,
            right=occ,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )
        rtpotmi = pd.merge(
            left=otmi,
            right=rt_pov,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )

        ybeasu = pd.merge(
            left=ybe,
            right=asu,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )
        ybeasu_bas = pd.merge(
            left=ybeasu,
            right=basics,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )

        allcrdi = pd.merge(
            left=rtpotmi,
            right=ybeasu_bas,
            how="left",
            left_on=merge_cols,
            right_on=merge_cols,
        )

        final_columns = [
            "NAME",
            "STATE",
            "COUNTY",
            "TRACT",
            "PLACE",
            "GEOID",
            "geometry",
            "total_population_universe",
            "median_household_income_2024$",
            "year_struct_built_2010_2019_percent",
            "year_struct_built_2020_later_percent",
            "tenure_renter_occupied_percent",
            "federal_poverty_limit_below_percent",
            "race_not_hispanic_white_alone_percent",
            "race_not_hispanic_black_alone_percent",
            "race_not_hispanic_aian_alone_percent",
            "race_not_hispanic_asian_alone_percent",
            "race_not_hispanic_nhpi_alone_percent",
            "race_not_hispanic_otherrace_alone_percent",
            "race_hispanic_percent",
            "race_personofcolor_percent",
            "sex_female_percent",
            "age_under_5_percent",
            "age_under_18_percent",
            "age_over_64_percent",
            "employment_civilian_labor_force_percent",
            "units_in_struct_occupied_housing_units",
            "units_in_struct_occupied_housing_units_1_detached",
            "units_in_struct_occupied_housing_units_1_attached",
            "units_in_struct_occupied_housing_units_2",
            "units_in_struct_occupied_housing_units_3_4",
            "units_in_struct_occupied_housing_units_5_9",
            "units_in_struct_occupied_housing_units_10_19",
            "units_in_struct_occupied_housing_units_20_49",
            "units_in_struct_occupied_housing_units_50_more",
            "units_in_struct_occupied_housing_units_mobile_home",
            "units_in_struct_occupied_housing_units_boat_rv_van_etc",
        ]

        allcrdi = allcrdi[final_columns]
        acs2parquet(allcrdi, str(in_path / f"acs_{geos}_{state}.parquet"))

In [ ]:
census_prep(STATE_CO, Path(r"C:\Users\MAtkinson\Documents\GitHub\basket\out"))